<table>
  <tr>
   <td width="220" align="center">
    <img src="https://anosys.ai/assets/img/3.png" width="200">
</td>
    <td valign="middle">
      <h1 style="margin:0;">Claude Agents - Observability Example</h1>
      <p style="margin:8px 0 0 0;">
        Interactive notebook demonstrating agentic workflows using the Anosys SDK and the Claude Agent SDK.
      </p>
    </td>
  </tr>
</table>


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/Anosys-AI/anosys-sdk/blob/main/examples/anosys_claude_agents_agentic_poc.ipynb)

This notebook demonstrates how to build a multi-agent system using the **Claude Agent SDK** and gain deep visibility into its behavior using **AnoSys Observability**. It is the Claude counterpart of the [OpenAI Agents PoC](https://github.com/Anosys-AI/anosys-sdk/blob/main/examples/anosys_openai_agentic_poc.ipynb).

The two SDKs model multi-agent systems differently: OpenAI Agents **hands off** control between agents, while a Claude agent **delegates** a task to a specialized **subagent** and gets the result back. The scenario is the same: a concierge routes questions to a weather specialist and a travel specialist.

### What you will learn:
1. How to define custom tools and specialized subagents.
2. How a main agent delegates work to subagents.
3. How to capture automatic traces (agent runs, tool calls, subagents, tokens and cost) with the AnoSys SDK.
4. How to associate traces with user context for better debugging.

## Step 1: Installation

First, we need to install the necessary libraries. We'll use the Claude Agent SDK for the logic and the AnoSys SDK for the observability. The Claude Agent SDK ships with the Claude Code runtime it drives, so there is nothing else to install.

Visit our library for the latest updates and features:
[anosys-sdk-claude-agents on PyPI](https://pypi.org/project/anosys-sdk-claude-agents)

In [ ]:
!pip install -q anosys-sdk-claude-agents claude-agent-sdk

## Step 2: Configuration

To run this PoC, you'll need two API keys:
1. **Anthropic API Key**: To power the agents.
2. **AnoSys API Key**: To send traces to your observability dashboard.

You can get your AnoSys API key from the [AnoSys Console](https://console.anosys.ai) > Data Ingestion > Integration Options.

In [ ]:
import os

try:
    from google.colab import userdata  # Use Colab secrets to store your keys
    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = userdata.get('ANOSYS_API_KEY')
except ImportError:
    # Running outside Colab — fall back to interactive input
    import getpass
    if "ANTHROPIC_API_KEY" not in os.environ:
        os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key: ")
    if "ANOSYS_API_KEY" not in os.environ:
        os.environ["ANOSYS_API_KEY"] = getpass.getpass("AnoSys API Key: ")

## Step 3: Initialize AnoSys Observability

`AnosysClaudeAgentsLogger` instruments the Claude Agent SDK's `query()` and `ClaudeSDKClient`. Once set up, every agent run, tool call, and subagent delegation will be captured as a trace, together with the run's token usage and cost.

Initialize it **before** importing `query`.

In [ ]:
from anosys_sdk_claude_agents import AnosysClaudeAgentsLogger

# Optional: Define a function to provide user context
def get_user_context():
    return {
        "session_id": "poc-session-001",
        "token": "demo-user-token"
    }

# Initialize once; agent_name is recorded as gen_ai.agent.name on the run span
AnosysClaudeAgentsLogger(get_user_context=get_user_context, agent_name="TriageAgent")

print("✅ AnoSys Observability initialized!")

## Step 4: Define Your Tools and Agents

In this scenario, we'll create three agents:
1. **weather-agent**: Subagent specialized in weather information.
2. **travel-agent**: Subagent specialized in flight and travel arrangements.
3. **TriageAgent**: The main agent, the entry point that delegates to the others.

**Note:** Custom tools are plain Python functions registered with the `@tool` decorator and served to Claude through an in-process MCP server. Each subagent is an `AgentDefinition` that only gets the tools it needs; Claude refers to them as `mcp__travel__<tool>`.

In [ ]:
import json
from claude_agent_sdk import (
    AgentDefinition, AssistantMessage, ClaudeAgentOptions, ResultMessage,
    ToolUseBlock, create_sdk_mcp_server, query, tool,
)

# --- Tools ---

@tool("get_weather", "Get the current weather for a city.", {"city": str})
async def get_weather(args):
    # Mock implementation
    print(f"[Tool] Calling get_weather for {args['city']}...")
    result = {"city": args["city"], "temperature": "22°C", "condition": "Sunny"}
    return {"content": [{"type": "text", "text": json.dumps(result)}]}

@tool("search_flights", "Search for flights between two cities.", {"origin": str, "destination": str})
async def search_flights(args):
    # Mock implementation
    print(f"[Tool] Calling search_flights from {args['origin']} to {args['destination']}...")
    result = [
        {"flight_id": "AN123", "price": "$450", "departure": "10:00 AM"},
        {"flight_id": "SY456", "price": "$380", "departure": "2:00 PM"}
    ]
    return {"content": [{"type": "text", "text": json.dumps(result)}]}

travel_tools = create_sdk_mcp_server(name="travel", tools=[get_weather, search_flights])

# --- Agents ---

subagents = {
    "weather-agent": AgentDefinition(
        description="Expert weather forecaster. Use for any question about the weather.",
        prompt="You are an expert weather forecaster. Use the get_weather tool to provide accurate info.",
        tools=["mcp__travel__get_weather"],
        model="haiku",
    ),
    "travel-agent": AgentDefinition(
        description="Travel specialist. Use for any question about flights or travel arrangements.",
        prompt="You help users find flights. Use the search_flights tool.",
        tools=["mcp__travel__search_flights"],
        model="haiku",
    ),
}

triage_options = ClaudeAgentOptions(
    system_prompt=(
        "You are a concierge. If the user asks about weather, delegate to the weather-agent. "
        "If they ask about flights, delegate to the travel-agent. Otherwise, answer as best as you can."
    ),
    model="claude-sonnet-5-5",
    mcp_servers={"travel": travel_tools},
    agents=subagents,
    allowed_tools=["Agent", "mcp__travel__get_weather", "mcp__travel__search_flights"],
    # Keep the demo self-contained: no shell, file or web access
    disallowed_tools=["Bash", "Read", "Write", "Edit", "Glob", "Grep", "WebFetch", "WebSearch", "NotebookEdit"],
    max_turns=10,
)

## Step 5: Execute the Agentic Flow

Now let's run a query that requires the TriageAgent to delegate the task to the travel-agent.

**Note:** `query()` is an async generator, and a notebook already runs an event loop, so we iterate it directly with `async for`.

In [ ]:
query_text = "I want to go from London to Tokyo. Can you check flights for me?"

print(f"Running query: {query_text}\n")

final_result = None
async for message in query(prompt=query_text, options=triage_options):
    if isinstance(message, AssistantMessage):
        for block in message.content:
            if isinstance(block, ToolUseBlock):
                who = "subagent" if message.parent_tool_use_id else "TriageAgent"
                print(f"[{who}] → {block.name} {json.dumps(block.input)[:120]}")
    elif isinstance(message, ResultMessage):
        final_result = message.result
        print(f"\n[run] {message.num_turns} turns, ${message.total_cost_usd or 0:.4f}")

print("\n--- Final Result ---")
print(final_result)

# Note: The traces are automatically being sent to AnoSys in the background!

## Step 6: Explore Your Traces in AnoSys

Head over to your **AnoSys Dashboard** to see the magic! You should see a new trace representing this execution (source tag `ClaudeAgentsTelemetry`).

### What to look for:
1. **Trace Timeline**: The `invoke_agent TriageAgent` run span, with every tool call as a child span.
2. **Subagent Delegation**: The `execute_tool Agent` span for the travel-agent (its `gen_ai_agent_name`), with the subagent's own `execute_tool mcp__travel__search_flights` call nested under it.
3. **Tool Execution**: Inspect the inputs and outputs of the `search_flights` tool.
4. **Usage & Cost**: Total tokens (including prompt-cache reads and writes) and the run's cost in USD on the run span.
5. **User Context**: Check the metadata to see the `session_id` we attached earlier.

### Troubleshooting
If you don't see traces immediately:
- Ensure your `ANOSYS_API_KEY` is correct. If the key is missing or invalid, you may see `405 Method Not Allowed` errors in the logs.
- Check the output logs in this notebook for any connection errors.
- The SDK sends data asynchronously; it might take a few seconds to appear.